# Weighted RRF Ensemble

Owner: Chua Rui Ning, Renee

This Colab notebook combines the selected LSTM and XGBoost rankings with fixed **75:25 LSTM:XGBoost weighted Reciprocal Rank Fusion (RRF)**. Validation and test are processed separately using each split’s top-200 rankings and labels.

## Outputs

The notebook writes one ensemble ranking and one manifest per split:

- Evaluation/ensemble/validation/rrf_lstm75_xgboost25_validation_top_200.parquet
- Evaluation/ensemble/test/rrf_lstm75_xgboost25_test_top_200.parquet

Each split’s folder also receives its matching manifest JSON.

## RRF definition

For an article, the ensemble score is the sum of each model’s contribution:

model_weight / (rank_constant + one_based_rank)

The LSTM weight is 0.75 and the XGBoost weight is 0.25. If an article is absent from a model’s top 200, that model contributes zero. The output contains the 100 highest fused-score articles per request.

The notebook checks that both model files cover exactly the requests in that split’s labels. The models must also use the same catalogue eligibility policy. Keep the weights fixed on test: use validation for model selection, then reserve test for final evaluation of the frozen pipeline.

In [1]:
# Mount Google Drive when running in Colab.
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder'
)
OUTPUT_ROOT = PROJECT_ROOT / 'Evaluation' / 'ensemble'

# Update run-folder names or filenames if the experiment exports move.
# The loader uses the first 100 articles from each list; longer lists are truncated.
LSTM_RUN = (
    PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run'
    / '20261104_175920_improved_seed42'
)
XGBOOST_RUN = (
    PROJECT_ROOT / 'Experiments' / '1.3_xgboost' / '20261105_small_seed42'
)

SPLITS = {
    'validation': {
        'labels': PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet',
        'rankings': {
            'LSTM': (
                LSTM_RUN / 'rankings' / 'validation' / 'submission_top_200.parquet'
            ),
            'XGBoost': (
                XGBOOST_RUN / 'rankings' / 'validation' / 'submission_top_200.parquet'
            ),
        },
    },
    'test': {
        'labels': PROJECT_ROOT / 'feature_engineering' / 'test_features.parquet',
        'rankings': {
            'LSTM': (
                LSTM_RUN / 'rankings' / 'test' / 'submission_top_200.parquet'
            ),
            'XGBoost': (
                XGBOOST_RUN / 'rankings' / 'test' / 'submission_top_200.parquet'
            ),
        },
    },
}

ENSEMBLE_NAME = 'rrf_lstm75_xgboost25'
MODEL_WEIGHTS = {'LSTM': 0.75, 'XGBoost': 0.25}
TOP_K = 200
RRF_RANK_CONSTANT = 60

if not np.isclose(sum(MODEL_WEIGHTS.values()), 1.0):
    raise ValueError('Model weights must sum to 1.')

print('Project root:', PROJECT_ROOT, PROJECT_ROOT.exists())
print('Output root:', OUTPUT_ROOT)
for split, config in SPLITS.items():
    print(f'{split} labels:', config['labels'], Path(config['labels']).is_file())
    for model_name, path in config['rankings'].items():
        print(f'{split} {model_name}:', path, Path(path).is_file())

Project root: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder True
Output root: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble
validation labels: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/validation_features.parquet True
validation LSTM: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/rankings/validation/submission_top_200.parquet True
validation XGBoost: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/1.3_xgboost/20261105_small_seed42/rankings/validation/submission_top_200.parquet True
test labels: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/test_features.parquet True
test LSTM: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/rankings/tes

In [3]:
def read_ranking_table(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {'.parquet', '.pq'}:
        frame = pd.read_parquet(path)
    elif suffix == '.csv':
        frame = pd.read_csv(path)
    else:
        raise ValueError(
            f'Unsupported ranking format: {path.suffix}; use Parquet or CSV'
        )

    required = {'behaviour_id', 'article_ids'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{path} must contain columns {sorted(required)}')
    if frame[['behaviour_id', 'article_ids']].isna().any().any():
        raise ValueError(f'{path} contains null request IDs or article lists')

    frame = frame[['behaviour_id', 'article_ids']].copy()
    frame['behaviour_id'] = frame['behaviour_id'].astype(str)
    if frame['behaviour_id'].duplicated().any():
        raise ValueError(f'{path} has duplicate behaviour_id rows')
    return frame


def parse_article_ids(value, *, model_name, behaviour_id, top_k):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f'{model_name}, {behaviour_id}: article_ids CSV value '
                'must be a JSON list'
            ) from exc

    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, (list, tuple)):
        raise ValueError(
            f'{model_name}, {behaviour_id}: article_ids must be a list'
        )

    numeric = pd.to_numeric(pd.Series(value), errors='raise')
    values = numeric.to_numpy(dtype='float64')
    invalid = (
        numeric.isna().any()
        or not np.isfinite(values).all()
        or (values % 1 != 0).any()
    )
    if invalid:
        raise ValueError(
            f'{model_name}, {behaviour_id}: article IDs must be finite integers'
        )
    if len(numeric) < top_k:
        raise ValueError(
            f'{model_name}, {behaviour_id}: expected at least {top_k} '
            f'candidates, got {len(numeric)}'
        )

    article_ids = numeric.iloc[:top_k].astype('int64').tolist()
    if len(set(article_ids)) != len(article_ids):
        raise ValueError(
            f'{model_name}, {behaviour_id}: duplicate IDs in top {top_k}'
        )
    return article_ids


def expected_behaviour_ids(path):
    labels = pd.read_parquet(
        path,
        columns=['user_id', 'impression_id'],
    )
    key_columns = ['user_id', 'impression_id']
    if labels[key_columns].isna().any().any():
        raise ValueError(f'{path} contains null user_id or impression_id')

    for column in key_columns:
        numeric = pd.to_numeric(labels[column], errors='raise')
        values = numeric.to_numpy(dtype='float64')
        if not np.isfinite(values).all() or (values % 1 != 0).any():
            raise ValueError(f'{path} {column} must contain finite integers')
        labels[column] = numeric.astype('int64')

    requests = labels[key_columns].drop_duplicates()
    return {
        f'{row.user_id}:{row.impression_id}'
        for row in requests.itertuples(index=False)
    }


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()


missing = []
for split, config in SPLITS.items():
    if not Path(config['labels']).is_file():
        missing.append(str(config['labels']))
    for model_name, path in config['rankings'].items():
        if not Path(path).is_file():
            missing.append(f'{split} {model_name}: {path}')

if missing:
    details = '\n'.join(missing)
    raise FileNotFoundError(f'Required input(s) not found:\n{details}')

In [4]:
outputs = {}

for split, config in SPLITS.items():
    print(f'\nCreating {split} ensemble...')
    expected_ids = expected_behaviour_ids(config['labels'])
    if not expected_ids:
        raise ValueError(f'{split} labels contain no requests')
    print(f'Expected {split} requests: {len(expected_ids):,}')

    loaded_inputs = []
    request_order = None

    for model_name in ('LSTM', 'XGBoost'):
        path = Path(config['rankings'][model_name])
        frame = read_ranking_table(path)
        request_ids = set(frame['behaviour_id'])

        if request_ids != expected_ids:
            missing_requests = expected_ids - request_ids
            extra_requests = request_ids - expected_ids
            raise ValueError(
                f'{split} {model_name} request coverage mismatch: '
                f'{len(missing_requests)} missing, '
                f'{len(extra_requests)} unexpected. '
                f'Missing examples: {list(sorted(missing_requests))[:5]}; '
                f'unexpected examples: {list(sorted(extra_requests))[:5]}'
            )

        if request_order is None:
            request_order = frame['behaviour_id'].tolist()

        request_map = {
            behaviour_id: parse_article_ids(
                value,
                model_name=model_name,
                behaviour_id=behaviour_id,
                top_k=TOP_K,
            )
            for behaviour_id, value in zip(
                frame['behaviour_id'],
                frame['article_ids'],
            )
        }
        loaded_inputs.append((model_name, path, request_map))
        print(
            f'{model_name}: validated {len(frame):,} requests; '
            f'using top {TOP_K}'
        )

    ensemble_rows = []
    for request_index, behaviour_id in enumerate(request_order, start=1):
        fused_scores = {}
        best_rank = {}

        for model_name, _, request_map in loaded_inputs:
            weight = MODEL_WEIGHTS[model_name]
            for rank, article_id in enumerate(
                request_map[behaviour_id],
                start=1,
            ):
                contribution = weight / (RRF_RANK_CONSTANT + rank)
                fused_scores[article_id] = (
                    fused_scores.get(article_id, 0.0) + contribution
                )
                best_rank[article_id] = min(
                    rank,
                    best_rank.get(article_id, rank),
                )

        fused_ids = sorted(
            fused_scores,
            key=lambda article_id: (
                -fused_scores[article_id],
                best_rank[article_id],
                article_id,
            ),
        )[:TOP_K]
        if len(fused_ids) != TOP_K:
            raise RuntimeError(
                f'{behaviour_id}: weighted RRF produced fewer than {TOP_K} articles'
            )

        ensemble_rows.append({
            'behaviour_id': behaviour_id,
            'article_ids': fused_ids,
        })
        if request_index % 5000 == 0:
            print(
                f'  fused {request_index:,}/{len(request_order):,} requests'
            )

    ensemble = pd.DataFrame(ensemble_rows)
    split_output_dir = OUTPUT_ROOT / split
    split_output_dir.mkdir(parents=True, exist_ok=True)

    output_path = (
        split_output_dir
        / f'{ENSEMBLE_NAME}_{split}_top_{TOP_K}.parquet'
    )
    ensemble.to_parquet(output_path, index=False)

    manifest = {
        'protocol': 'weighted_rrf_v1',
        'split': split,
        'request_count': int(len(ensemble)),
        'input_top_k_per_model': TOP_K,
        'output_top_k': TOP_K,
        'rrf_rank_constant': RRF_RANK_CONSTANT,
        'weights': MODEL_WEIGHTS,
        'score_definition': (
            'sum(weight_model / '
            '(rrf_rank_constant + one_based_rank))'
        ),
        'missing_from_model': 'zero contribution',
        'tie_break': 'best contributing rank, then ascending article_id',
        'labels': {
            'path': str(config['labels']),
            'sha256': sha256_file(config['labels']),
        },
        'inputs': [
            {
                'model': model_name,
                'path': str(path),
                'sha256': sha256_file(path),
            }
            for model_name, path, _ in loaded_inputs
        ],
        'output': {
            'path': str(output_path),
            'sha256': sha256_file(output_path),
        },
    }

    manifest_path = (
        split_output_dir
        / f'{ENSEMBLE_NAME}_{split}_top_{TOP_K}.manifest.json'
    )
    manifest_path.write_text(
        json.dumps(manifest, indent=2) + '\n',
        encoding='utf-8',
    )
    outputs[split] = {
        'ranking': output_path,
        'manifest': manifest_path,
        'requests': len(ensemble),
    }

    lengths = ensemble['article_ids'].map(len)
    print('Created ensemble:', output_path)
    print('Created manifest:', manifest_path)
    print(f'Requests: {len(ensemble):,}')
    print(
        'Candidates per request: '
        f'{lengths.min()}-{lengths.max()}'
    )

print('\nFinished validation and test ensemble generation.')
for split, result in outputs.items():
    print(
        split,
        '=>',
        result['ranking'],
        '| manifest:',
        result['manifest'],
    )


Creating validation ensemble...
Expected validation requests: 29,110
LSTM: validated 29,110 requests; using top 200
XGBoost: validated 29,110 requests; using top 200
  fused 5,000/29,110 requests
  fused 10,000/29,110 requests
  fused 15,000/29,110 requests
  fused 20,000/29,110 requests
  fused 25,000/29,110 requests
Created ensemble: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/validation/rrf_lstm75_xgboost25_validation_top_200.parquet
Created manifest: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/ensemble/validation/rrf_lstm75_xgboost25_validation_top_200.manifest.json
Requests: 29,110
Candidates per request: 200-200

Creating test ensemble...
Expected test requests: 29,142
LSTM: validated 29,142 requests; using top 200
XGBoost: validated 29,142 requests; using top 200
  fused 5,000/29,142 requests
  fused 10,000/29,142 requests
  fused 15,000/29,142 requests
  fused 20,000/29,142 requests
  fused 25,000/29,14

# Output and next steps

For each split, the notebook saves:

- the weighted RRF top-200 Parquet;
- a manifest containing the split, weights, input/output paths, and file hashes.

Use the **validation** ranking for model comparison and reranker development. Use the **test** ranking only for final evaluation after the complete pipeline is frozen. Do not use test results to change weights or other settings.